# Manual de ejecución de los modelos de tesis

## Detección de polaridad emocional en voz de adultos mayores mediante modelos de aprendizaje automático

Este notebook funciona como guía para reproducir los experimentos desarrollados en la tesis. Incluye la preparación del entorno, la organización y obtención de los conjuntos de datos, la ejecución de los modelos y la evaluación de resultados.

**Clases de polaridad:** positiva, neutral y negativa.

> **Nota:** Las rutas y enlaces de los datasets deben sustituirse por las ubicaciones exactas utilizadas en la versión final de los experimentos.

## 1. Contenido del manual

1. Requisitos de ejecución
2. Configuración de Google Colab
3. Instalación de dependencias
4. Organización de los datasets
5. Preparación de los datos
6. Modelo WSREmotionDetection
7. Modelo WRSEmotioMFCC/PLP
8. Experimentos entre corpus
9. Evaluación mediante métricas
10. Guardado de resultados
11. Problemas frecuentes


## 2. Requisitos de ejecución

Los experimentos pueden ejecutarse en **Google Colab**. Se recomienda utilizar una sesión con GPU para reducir el tiempo de entrenamiento, especialmente en los experimentos con WRSEmotionDetection y validación cruzada.

### Recomendaciones
- Cuenta de Google con acceso a Google Colab.
- GPU habilitada en Colab.
- Conexión a Internet para instalar dependencias y obtener los datasets cuando corresponda.
- Espacio suficiente de almacenamiento para los archivos de audio y modelos generados.

## 3. Configuración de Google Colab

En Google Colab, seleccionar:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.

La siguiente celda permite comprobar si la GPU está disponible.

### Recomendación
Manejar GPU con mayor capacidad de preferencia GPU A100.

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Instalación de dependencias

Instalar las librerías utilizadas por los experimentos. Las versiones definitivas deben coincidir con las empleadas para generar los resultados reportados en la tesis.

In [ ]:
!pip install -q transformers datasets torchaudio librosa scikit-learn accelerate torchcodec

## 5. Organización de los datasets

Los experimentos utilizan conjuntos de datos de habla emocional. Antes de ejecutar los modelos, se recomienda organizar los archivos de forma que cada corpus pueda identificarse y procesarse de manera independiente.

### Datasets utilizados

| Dataset | Uso principal |
|---|---|
| **EmoMatchSpanishDB (EMOMATCH)** | Entrenamiento/evaluación de modelos en español |
| **Corpus propio** | Datos recopilados para la investigación con adultos mayores |
| **WRSEmotion** | Evaluación externa / generalización |

### Ubicación

Se recomienda descargar previamente los conjuntos de datos correspondientes a cada corpus desde la carpeta WSREmotion disponible en Google Drive. Durante la ejecución de cada experimento, los archivos del corpus que se utilizará deberán copiarse y organizarse dentro del entorno de ejecución de Google Colab, creando una carpeta independiente para cada corpus.

De esta manera, cada notebook contará con los conjuntos de datos necesarios para ejecutar el modelo correspondiente, manteniendo una estructura de archivos organizada y facilitando la correcta definición de las rutas de acceso durante el procesamiento.

Por ejemplo, dentro del entorno de Colab se puede utilizar una estructura como la siguiente:


```text
/content/
  ├── EMOMATCH/
  ├── MESD/
  └── WRSEmotion/
```

**Ruta del drive:** https://drive.google.com/drive/folders/1rCye3N4Dr6Bin-Hkwigm7KC-2fJSm2-q?usp=drive_link


** Nota: ** En caso de no contar con acceso a la carpeta, será necesario solicitar al propietario los permisos correspondientes para acceder a la carpeta de Google Drive.

## 6. Ruta de los datasets para la ejecución de los modelos

Debido a que los datasets se encuentran almacenados dentro de carpetas del entorno de ejecución, es necesario especificar correctamente la ruta de acceso a los archivos en cada uno de los modelos.

La ruta deberá colocarse de la siguiente manera en todos los modelos:



In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MESD_PATH = "/content/Carpeta_dataset"

## 7. Parámetros generales

Para reproducir los experimentos se deben conservar los parámetros definidos en la tesis.

In [ ]:
SEED = 42
SAMPLE_RATE = 16000
NUM_LABELS = 3
K_FOLDS = 5

print('SEED =', SEED)
print('SAMPLE_RATE =', SAMPLE_RATE)
print('NUM_LABELS =', NUM_LABELS)
print('K_FOLDS =', K_FOLDS)

## 8. Preparación y partición de los datos

La partición de los datos debe realizarse de manera estratificada para conservar la proporción de las clases. El esquema documentado para los experimentos es:

- **70 %** entrenamiento.
- **10 %** validación.
- **20 %** prueba.

El conjunto de prueba debe mantenerse separado de las decisiones de optimización del modelo.

Cuando se utilice validación cruzada, debe conservarse la semilla establecida y la separación entre entrenamiento y evaluación.

## 9. Preprocesamiento del audio

El flujo de procesamiento contempla, según el experimento:

1. Carga del archivo de audio.
2. Conversión/remuestreo a **16 kHz**.
3. Normalización.
4. Detección de actividad de voz (VAD), cuando corresponda.
5. Segmentación y preparación de las señales.
6. Extracción de características para el modelo clásico o preparación de la señal para WSREmotionDetection.

## 10. Modelo WSREmotionDetection

WSREmotionDetection se utiliza como modelo basado en Transformers para obtener representaciones de la señal de voz y realizar la clasificación de las tres clases de polaridad.

### Flujo de ejecución

```text
Audio → preprocesamiento → Wav2Vec2Processor → Wav2Vec2
→ clasificación → predicciones → métricas
```

En esta sección debe colocarse el notebook/código definitivo de entrenamiento utilizado en la tesis. Se recomienda conservar los mismos parámetros de entrenamiento, división de datos, semilla y modelo preentrenado para reproducir los resultados.

In [ ]:
# Ejemplo de importaciones utilizadas por el experimento WSREmotionDetection.
# Sustituir/acompañar esta celda con el código definitivo del entrenamiento.

import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification


    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



### Resultados de WRSEmotionDetection

Después del entrenamiento se deben calcular y registrar:

- Accuracy.
- Precision macro.
- Recall macro.
- F1-score macro.


Los resultados deben guardarse indicando claramente el dataset, configuración y experimento correspondiente.

## 11. Modelo WRSEmotionMFCC/PLP

Este enfoque utiliza características acústicas extraídas de la señal de voz y un clasificador SVM.

### Flujo de ejecución

```text
Audio → preprocesamiento → MFCC + PLP
→ concatenación de características → SVM
→ predicciones → métricas
```

En la configuración utilizada en los experimentos se consideran características MFCC y PLP. El código definitivo empleado en la tesis debe incorporarse en las celdas siguientes.

In [ ]:
# Importaciones principales para el enfoque MFCC + PLP + SVM.
import numpy as np
import librosa
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

### Parámetros de características

La configuración reportada contempla **13 coeficientes MFCC** y un banco Mel con **26 filtros**, además de las características PLP utilizadas por el pipeline correspondiente.

## 12. Experimentos entre corpus

Para evaluar la capacidad de generalización, se contempla un escenario en el que el modelo se entrena con los datos disponibles para entrenamiento y posteriormente se evalúa sobre un corpus externo.

Un experimento documentado es:

**Entrenamiento:** EMOMATCH + WRSEmotion  
**Evaluación externa:** WRSEmotion

El conjunto externo no debe utilizarse durante el ajuste del modelo.

## 13. Métricas de evaluación

Para comparar los modelos se utilizan las siguientes métricas:

- **Accuracy:** proporción total de predicciones correctas.
- **Precision:** proporción de predicciones positivas que son correctas.
- **Recall:** proporción de ejemplos de una clase que el modelo identifica correctamente.
- **F1-score:** media armónica entre precision y recall.

Se recomienda reportar las métricas con promedio **macro**, ya que cada clase recibe el mismo peso independientemente de su cantidad de ejemplos.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Ejemplo de evaluación.
# Sustituir y_true/y_pred por las etiquetas reales y las predicciones del modelo.
# y_true = [...]
# y_pred = [...]

# accuracy = accuracy_score(y_true, y_pred)
# precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
# recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
# f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

# print(f'Accuracy: {accuracy:.4f}')
# print(f'Precision macro: {precision:.4f}')
# print(f'Recall macro: {recall:.4f}')
# print(f'F1 macro: {f1:.4f}')

## 14. Baselines

Para contextualizar el desempeño de los modelos, se pueden calcular dos referencias:

1. **Baseline aleatorio:** predicciones realizadas de manera aleatoria entre las clases.
2. **Baseline de clase mayoritaria:** asignación de la clase con mayor frecuencia en el conjunto de entrenamiento.

Los baselines permiten comprobar que el desempeño obtenido por los modelos es superior a estrategias sencillas.

## 15. Guardado de resultados

Para facilitar la reproducibilidad, se recomienda guardar los resultados en archivos independientes por experimento.

Una estructura sugerida es:

```text
results/
├── wav2vec2/
│   ├── emomatch/
│   └── wrse_motion/
├── mfcc_plp_svm/
└── baselines/
```

Cada resultado debe incluir, como mínimo, el nombre del dataset, modelo, semilla, partición y métricas obtenidas.

## 16. Lista de comprobación antes de ejecutar un experimento

- [ ] GPU habilitada cuando sea necesaria.
- [ ] Dependencias instaladas.
- [ ] Datasets disponibles en las rutas indicadas.
- [ ] Etiquetas correctamente asignadas.
- [ ] Frecuencia de muestreo establecida en 16 kHz.
- [ ] Semilla establecida en 42.
- [ ] División de datos definida antes del entrenamiento.
- [ ] Conjunto de prueba separado del ajuste.
- [ ] Métricas configuradas con promedio macro.
- [ ] Carpeta de resultados disponible.


## 17. Problemas frecuentes

### No aparece la GPU
Verificar el tipo de entorno de ejecución en Google Colab y volver a ejecutar la celda de comprobación.

### No se encuentra un dataset
Comprobar que la carpeta se encuentre en la ruta definida en `DATA_PATH` y que los nombres de archivos coincidan con los utilizados por el código.

### Error por librería faltante
Ejecutar nuevamente la celda de instalación de dependencias y reiniciar el entorno si Colab lo solicita.

### Falta de memoria
Reducir el tamaño del lote, procesar los audios por partes o utilizar una GPU con mayor memoria, manteniendo documentado cualquier cambio experimental.

### El experimento tarda demasiado
La validación cruzada puede incrementar considerablemente el tiempo de ejecución, especialmente en modelos basados en Transformers. No cerrar la sesión mientras el entrenamiento esté ejecutándose y guardar los checkpoints cuando el código lo permita.

## 18. Reproducibilidad

Para obtener resultados comparables con los reportados en la tesis, se deben conservar las mismas versiones de las librerías, datasets, particiones, semillas, parámetros de entrenamiento y procedimiento de evaluación.

Cualquier modificación realizada posteriormente debe registrarse como una configuración o experimento diferente.

# Fin del manual

Este notebook constituye una guía de ejecución y documentación de los experimentos. Antes de utilizarlo como versión definitiva, deben incorporarse las rutas y enlaces exactos de los datasets y el código final de cada experimento.